##**Notebook PC#11**

## Encoder-Decoder LSTM for Natural Language Processing. Code based on the content of [this book](https://machinelearningmastery.com/lstms-with-python/) in its Chapter 9.

Professor: Fernando J. Von Zuben <BR>
Cursos: IA353A / EG453A (FEEC/Unicamp) - 1s2026 <BR>
Aluno(a): João Lucas Morais Ortiz RA: 297611<BR>
Aluno(a): Daniele Souza Gonçalves RA: 24802<BR>

In [1]:
from random import seed
from random import randint
from numpy import array
from numpy import argmax

In [2]:
def random_sum_pairs(n_examples, n_numbers, largest):
    X,y = list(), list()
    for i in range(n_examples):
        in_pattern=[randint(1,largest) for _ in range(n_numbers)]
        out_pattern = sum(in_pattern)
        X.append(in_pattern)
        y.append(out_pattern)
    return X,y

In [3]:
seed(1)
n_samples =1
n_numbers = 2
largest = 10
X,y = random_sum_pairs(n_samples, n_numbers, largest)
print(X,y)

[[3, 10]] [13]


In [4]:
from math import ceil
from math import log10

In [5]:
def to_string(X,y,n_numbers,largest):
    max_length = n_numbers*ceil(log10(largest+1)) + n_numbers - 1
    Xstr = list()
    for pattern in X:
        strp = '+'.join([str(n) for n in pattern])
        strp = ''.join([' ' for _ in range(max_length-len(strp))]) + strp
        Xstr.append(strp)
    maxlength = ceil(log10(n_numbers*(largest+1)))
    ystr = list()
    for pattern in y:
        strp = str(pattern)
        strp = ''.join([' 'for _ in range(maxlength-len(strp))]) + strp
        ystr.append(strp)
    return Xstr, ystr

In [6]:
seed(1)
n_samples = 1
n_numbers = 2
largest = 10

In [7]:
X,y = random_sum_pairs(n_samples, n_numbers, largest)
print(X,y)

X,y = to_string(X,y,n_numbers,largest)
print(X,y)

[[3, 10]] [13]
[' 3+10'] ['13']


In [10]:
alphabet = ['0','1','2','3','4','5','6','7','8','9','+',' ']

In [11]:
def integer_encode(X,y,alphabet):
    char_to_int = dict((c,i) for i,c in enumerate(alphabet))
    Xenc = list()
    for pattern in X:
        integer_encoded = [char_to_int[char] for char in pattern]
        Xenc.append(integer_encoded)
    yenc = list()
    for pattern in y:
        integer_encoded = [char_to_int[char] for char in pattern]
        yenc.append(integer_encoded)
    return Xenc, yenc

In [12]:
X,y = integer_encode(X,y,alphabet)

In [13]:
print(X,y)

[[11, 3, 10, 1, 0]] [[1, 3]]


In [14]:
def one_hot_encode(X,y,max_int):
    Xenc = list()
    for seq in X:
        pattern = list()
        for index in seq:
            vector = [0 for _ in range(max_int)]
            vector[index] = 1
            pattern.append(vector)
        Xenc.append(pattern)

    yenc = list()
    for seq in y:
        pattern = list()
        for index in seq:
            vector = [0 for _ in range(max_int)]
            vector[index] = 1
            pattern.append(vector)
        yenc.append(pattern)
    return Xenc, yenc

In [15]:
X,y = one_hot_encode(X,y,len(alphabet))

In [16]:
print(X,y)

[[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]] [[[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]]]


In [17]:
def generate_data(n_samples,n_numbers, largest, alphabet):
    X,y = random_sum_pairs(n_samples,n_numbers,largest)
    X,y = to_string(X,y,n_numbers,largest)
    X,y = integer_encode(X,y,alphabet)
    X,y = one_hot_encode(X,y,len(alphabet))
    X,y = array(X), array(y)
    return X,y

In [19]:
def invert(seq,alphabet):
    int_to_char = dict((i,c) for i,c in enumerate(alphabet))
    strings  = list()
    for pattern in seq:
        string = int_to_char[argmax(pattern)]
        strings.append(string)
    return ''.join(strings)

In [20]:
n_terms = 3
largest = 10
alphabet = [str(x) for x in range(10)] + ['+', ' ']

In [21]:
n_chars = len(alphabet)
n_in_seq_length = n_terms*ceil(log10(largest+1)) +n_terms-1
n_out_seq_length = ceil(log10(n_terms*(largest+1)))

In [22]:
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import RepeatVector
from keras.layers import TimeDistributed
from keras.layers import Dense, Input

In [23]:
model = Sequential()
model.add(Input(shape=(n_in_seq_length,n_chars)))
model.add(LSTM(75))
model.add(RepeatVector(n_out_seq_length))
model.add(LSTM(50,return_sequences=True))
model.add(TimeDistributed(Dense(n_chars,activation='softmax')))
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 75)             │        26,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 2, 75)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 2, 50)          │        25,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 2, 12)          │           612 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,212 (203.95 KB)

 Trainable params: 52,212 (203.95 KB)

 Non-trainable params: 0 (0.00 B)

None


In [24]:
X,y = generate_data(75000,n_terms,largest,alphabet)
model.fit(X,y,epochs=1,batch_size=10)

7500/7500 ━━━━━━━━━━━━━━━━━━━━ 55s 6ms/step - accuracy: 0.8839 - loss: 0.3883


In [25]:
X,y = generate_data(100,n_terms,largest,alphabet)
loss,acc = model.evaluate(X,y,verbose=0)
print('Loss: %f, Accuracy: %f' %(loss,acc*100))

Loss: 0.041022, Accuracy: 99.500000


In [ ]:
for _ in range(10):
    X,y = generate_data(1,n_terms,largest,alphabet)
    yhat = model.predict(X,verbose=0)
    in_seq = invert(X[0],alphabet)
    out_seq = invert(y[0],alphabet)
    predicted = invert(yhat[0],alphabet)
    print('%s = %s (expect %s)' %(in_seq,predicted,out_seq))

  5+2+10 = 17 (expect 17)
   6+4+5 = 15 (expect 15)
   9+2+2 = 13 (expect 13)
  4+10+7 = 21 (expect 21)
 10+10+2 = 22 (expect 22)
   3+8+4 = 15 (expect 15)
   1+8+1 = 10 (expect 10)
   3+5+9 = 17 (expect 17)
   6+1+5 = 12 (expect 12)
   8+2+2 = 12 (expect 12)


<font color="green">
Atividade (a) <br>
Como são gerados os dados de treinamento?
</font>

**Resposta:**

Os dados de treinamento são gerados por um pipeline de 4 etapas que transforma operações aritméticas em sequências adequadas para o modelo seq2seq:

1. **Geração de pares numéricos** (`random_sum_pairs`): Cria aleatoriamente listas de `n_numbers` inteiros no intervalo [1, largest] e calcula suas somas. Ex: entrada `[3, 10]`, saída `13`.

2. **Conversão para strings** (`to_string`): Transforma os números em formato textual com padding à esquerda usando espaços, garantindo tamanho fixo. O comprimento da entrada é `n_numbers * ceil(log10(largest+1)) + n_numbers - 1` (para acomodar os dígitos e símbolos '+'). Ex: `" 3+10"` → `"13"`.

3. **Codificação inteira** (`integer_encode`): Mapeia cada caractere para um índice usando o alfabeto `['0'-'9', '+', ' ']` (12 símbolos). Cada caractere vira um número de 0 a 11.

4. **One-hot encoding** (`one_hot_encode`): Converte cada índice inteiro em um vetor binário de tamanho 12 (tamanho do alfabeto), onde apenas a posição correspondente ao caractere é 1.

O resultado são tensores 3D: **X** com shape `(n_samples, comprimento_entrada, 12)` e **y** com shape `(n_samples, comprimento_saída, 12)`, representando as sequências de entrada e saída como matrizes de one-hot vectors. Este formato permite que o modelo trate a operação matemática como um problema de tradução sequência-para-sequência.

<font color="green">
Atividade (b) <br>
Como uma calculadora simples pode operar baseada no conceito de tradução de frases, ou seja, sem realizar operações algébricas?
</font>

**Resposta:**

A rede neural **não realiza operações algébricas** — ela trata a soma como um problema de **tradução de sequências** (seq2seq), similar à tradução entre idiomas:

1. **Analogia com tradução**: A entrada `"3+10"` é uma "frase" no idioma de origem, e `"13"` é a "tradução" correspondente. O modelo aprende estatisticamente a mapear padrões de caracteres de entrada para padrões de saída.

2. **Arquitetura encoder-decoder**:
   - **Encoder LSTM**: Processa a sequência de entrada caractere por caractere, comprimindo toda a informação em um vetor de contexto (estado oculto final).
   - **RepeatVector**: Replica este vetor de contexto para cada posição da sequência de saída.
   - **Decoder LSTM**: Gera a sequência de saída, produzindo uma distribuição de probabilidade sobre os 12 caracteres possíveis via softmax para cada posição.

3. **Aprendizado por padrões estatísticos**: O modelo memoriza/generaliza correlações entre sequências de entrada e saída dos 75.000 exemplos de treinamento, sem "entender" o conceito matemático de adição.

**O modelo realmente aprende a fazer contas?** Na verdade, **não**. Ele aprende a identificar qual é a resposta estatisticamente mais provável dado um padrão de entrada. Para verificar isso, podemos propor um exercício mental: se um ser humano, ou um sistema que sabe fazer computação como uma ULA, uma máquina calculadora, etc... aprendesse a somar como essa máquina LSTM aprendeu, dado que ela tivesse capacidade (em números de bits de representação), ela generalizaria a regra para qualquer soma. E mesmo que ela não tivesse, ela faria a soma corretamente até dentro das suas capacidades, mesmo com o resultado ficando errado por conta de overflow. Entretanto, a LSTM mesmo tendo a capacidade de receber uma entrada, ela as vezes não tem mapeado a saída correta como a mais probabilisticamente provavel. demonstro isso nos experimentos abaixo. No primeiro, tento fazer esse overflow: o modelo foi treinado com 3 números, tentei fazer ele trabalhar com 4. No segundo, usando 3 números mesmo, mostro como é possível confundir o modelo com algumas somas.

In [ ]:

# Experimento: testar com 9+9+9+9 (soma = 36, fora do range de treinamento pois tem n_terms=4)
# O modelo foi treinado com n_terms=3 e largest=10

# Criar entrada manualmente usando as funções existentes
seed(42)  # seed diferente para não repetir dados de treino
X_test, y_test = generate_data(1, n_terms, largest, alphabet)

# Agora vamos forçar uma entrada específica: 9+9+9
# Primeiro, criar no formato correto
X_manual = [[9, 9, 9, 9]]
y_manual = [36]

X_str, y_str = to_string(X_manual, y_manual, n_terms, largest)
print(f"Entrada string: '{X_str[0]}' -> Saída esperada: '{y_str[0]}'")

X_enc, y_enc = integer_encode(X_str, y_str, alphabet)
X_oh, y_oh = one_hot_encode(X_enc, y_enc, len(alphabet))
X_final = array(X_oh)

# Fazer predição
yhat = model.predict(X_final, verbose=0)
predicted = invert(yhat[0], alphabet)

print(f"Entrada: {X_str[0]}")
print(f"Predição do modelo: '{predicted}'")
print(f"Resposta correta: '{y_str[0]}'")

Entrada string: ' 9+9+9+9' -> Saída esperada: '36'
Entrada:  9+9+9+9
Predição do modelo: '29'
Resposta correta: '36'


In [35]:
# Outros testes com valores dentro do treinamento,, tentando fazer o modelo se confundir

test_cases = [
    ([1, 1, 1], 3),
    ([10, 10, 10], 30),  # limite máximo
    ([5, 5, 5], 15),
    ([0, 0, 0], 0),
    ([9, 8, 7], 24),
]

print("Testando generalização do modelo:\n")
for X_manual, expected in test_cases:
    y_manual = [expected]
    X_str, y_str = to_string([X_manual], y_manual, n_terms, largest)
    X_enc, y_enc = integer_encode(X_str, y_str, alphabet)
    X_oh, y_oh = one_hot_encode(X_enc, y_enc, len(alphabet))
    X_final = array(X_oh)
    
    yhat = model.predict(X_final, verbose=0)
    predicted = invert(yhat[0], alphabet)
    
    status = "✓" if predicted.strip() == y_str[0].strip() else "✗"
    print(f"{status} {X_str[0]} = {predicted.strip()} (esperado: {y_str[0].strip()})")

Testando generalização do modelo:

✗    1+1+1 = 5 (esperado: 3)
✗ 10+10+10 = 29 (esperado: 30)
✓    5+5+5 = 15 (esperado: 15)
✗    0+0+0 = 28 (esperado: 0)
✓    9+8+7 = 24 (esperado: 24)
